# 🚀 High-Speed EVA-02 9-Tile Keyframe Batch Encoder (Colab G4 / GPU)
Encodes 10,000–150,000+ video contact sheets into 9-tile MIL embeddings using Google Colab G4 (Blackwell RTX Pro 6000) or any high-VRAM GPU (A100 / L4 / T4).

### Instructions:
1. **Runtime Type**: In Colab menu, choose **Runtime -> Change runtime type -> G4 GPU (or A100 / L4 / T4)**.
2. Upload your batch archive (`colab_batch_all.tar`, `colab_batch.tar.gz`, etc.) and optional fine-tuned checkpoint (`best.pt`) to your Google Drive root (or directly to `/content/`).
3. Run all cells below.

In [ ]:
# Step 1: Install Dependencies & Check GPU / NVMe Disk
!nvidia-smi
!df -h /content
!pip install -q open-clip-torch timm torchvision pillow tqdm

In [ ]:
# Step 2: Locate Archive, Auto-Detect Format (by Magic Bytes), & Extract to NVMe
import os, sys, tarfile, shutil, subprocess
from tqdm.auto import tqdm

# Attempt to mount Google Drive if available
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    print("[+] Google Drive mounted successfully.")
except Exception as e:
    print(f"[*] Note: Drive mount skipped or not interactive ({e}). Continuing with local storage...")

# Search candidate paths for any batch archive (.tar, .tar.gz, or .tgz)
TAR_PATH = None
search_folders = ['/content/drive/MyDrive', '/content', '.']
for folder in search_folders:
    if os.path.exists(folder):
        for fname in sorted(os.listdir(folder)):
            if ('colab_batch' in fname or 'mosaic' in fname) and (fname.endswith('.tar') or fname.endswith('.tar.gz') or fname.endswith('.tgz')):
                TAR_PATH = os.path.join(folder, fname)
                break
    if TAR_PATH:
        break

if TAR_PATH is None:
    raise FileNotFoundError(
        "Could not find any batch archive (colab_batch*.tar or .tar.gz)! "
        "Please upload it to /content/ via the left file browser, or copy it to Google Drive root."
    )

# Inspect magic bytes instead of trusting file extension
# (Handles uncompressed .tar archives even if renamed to .tar.gz!)
with open(TAR_PATH, 'rb') as f:
    magic = f.read(2)
is_gzip = (magic == b'\x1f\x8b')

tar_mb = os.path.getsize(TAR_PATH) / (1024 * 1024)
print(f"[*] Found archive: {TAR_PATH} ({tar_mb:.1f} MB)")
print(f"[*] Detected format: {'gzip-compressed tar' if is_gzip else 'uncompressed tar'} (magic={magic!r})")
print("[*] Extracting to local NVMe scratchpad (/content/work)... (this may take 2-4 mins for ~12 GB over Drive)")

os.makedirs('/content/work', exist_ok=True)

tar_ok = False
if shutil.which('tar'):
    try:
        # Explicitly select gzip (-z) ONLY when magic bytes match gzip, preventing suffix confusion
        cmd = ['tar', '-zxf' if is_gzip else '-xf', TAR_PATH, '-C', '/content/work']
        subprocess.run(cmd, check=True)
        tar_ok = True
    except Exception as e:
        print(f"[!] Native tar command returned error ({e}); falling back to Python tarfile...")

if not tar_ok:
    mode = 'r:gz' if is_gzip else 'r:'
    with tarfile.open(TAR_PATH, mode, bufsize=4 * 1024 * 1024) as tar:
        extract_kwargs = {'filter': 'data'} if sys.version_info >= (3, 12) else {}
        for member in tqdm(tar, desc='Extracting', unit='file'):
            tar.extract(member, path='/content/work', **extract_kwargs)

print("[+] Extraction complete! Workspace contents:", os.listdir('/content/work'))

In [ ]:
# Step 3: High-Throughput 9-Tile Batch Encoder on Tensor Cores (Streaming Memmap)
import json, time, os, shutil
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.io import decode_jpeg, ImageReadMode
from PIL import Image
import open_clip

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# Locate manifest and mosaics folder
manifest_file = '/content/work/manifest.json'
mosaics_dir = '/content/work/mosaics'
if not os.path.exists(manifest_file):
    for root, dirs, files in os.walk('/content/work'):
        if 'manifest.json' in files:
            manifest_file = os.path.join(root, 'manifest.json')
            mosaics_dir = os.path.join(root, 'mosaics')
            break

if not os.path.exists(manifest_file):
    raise FileNotFoundError("manifest.json not found in /content/work! Check Step 2 extraction output.")

with open(manifest_file, 'r', encoding='utf-8') as f:
    items = json.load(f)

N = len(items)
print(f"[*] Loaded manifest with {N:,} items to encode on {DEVICE}.")

# Dynamic batch sizing scaled for G4 Blackwell (96 GB) down to T4 (16 GB)
if DEVICE == 'cuda':
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    if total_vram_gb >= 45.0:
        BATCH_SIZE = 64
    elif total_vram_gb >= 20.0:
        BATCH_SIZE = 36
    elif total_vram_gb >= 10.0:
        BATCH_SIZE = 16
    else:
        BATCH_SIZE = 8
    print(f"[*] GPU: {gpu_name} ({total_vram_gb:.1f} GB VRAM) -> Batch Size: {BATCH_SIZE} videos ({BATCH_SIZE * 9} tiles/step)")
else:
    BATCH_SIZE = 2
    print(f"[*] Running on CPU -> Batch Size: {BATCH_SIZE}")

class ColabMosaicDataset(Dataset):
    def __init__(self, item_list, img_dir):
        self.items = item_list
        self.img_dir = img_dir

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        entry = self.items[idx]
        img_path = os.path.join(self.img_dir, entry['filename'])
        try:
            with open(img_path, 'rb') as f:
                raw_bytes = bytearray(f.read())
            try:
                raw_tensor = torch.frombuffer(raw_bytes, dtype=torch.uint8)
                im = decode_jpeg(raw_tensor, mode=ImageReadMode.RGB)
            except Exception:
                # Fallback to PIL if asset isn't standard baseline JPEG (e.g. WebP/PNG)
                pil_im = Image.open(img_path).convert('RGB')
                im = torch.from_numpy(np.array(pil_im)).permute(2, 0, 1).contiguous()
            _, h, w = im.shape
            tw, th = w // 3, h // 3
            crops = torch.stack([
                im[:, r*th:(r+1)*th, c*tw:(c+1)*tw] for r in range(3) for c in range(3)
            ], dim=0)
            resized = F.interpolate(crops.float(), size=(336, 336), mode='bilinear', align_corners=False).to(torch.uint8)
        except Exception:
            resized = torch.zeros(9, 3, 336, 336, dtype=torch.uint8)
        return resized, idx

def collate_crops(batch):
    tensors, indices = zip(*batch)
    return torch.cat(tensors, dim=0), list(indices)

num_workers = min(8, os.cpu_count() or 2)
dataset = ColabMosaicDataset(items, mosaics_dir)
loader_kwargs = {'num_workers': num_workers, 'pin_memory': (DEVICE == 'cuda')}
if num_workers > 0:
    loader_kwargs['prefetch_factor'] = 2
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_crops, **loader_kwargs)

# Load EVA-02 backbone
print("[*] Loading EVA-02-CLIP-L-14-336...")
model, _, _ = open_clip.create_model_and_transforms('hf-hub:timm/eva02_large_patch14_clip_336.merged2b_s6b_b61k', device=DEVICE)

# Check for optional custom fine-tuned weights BEFORE casting to FP16
CHECKPOINT_CANDIDATES = [
    '/content/drive/MyDrive/best.pt',
    '/content/drive/MyDrive/eva02_curated_v2.pt',
    '/content/best.pt',
    '/content/eva02_curated_v2.pt',
    '/content/work/best.pt',
    '/content/work/model.pt'
]
loaded_custom_ckpt = False
for wpath in CHECKPOINT_CANDIDATES:
    if os.path.exists(wpath):
        print(f"[+] Found custom fine-tuned weights at {wpath}! Loading state dict...")
        try:
            ckpt = torch.load(wpath, map_location='cpu', weights_only=True)
        except Exception:
            ckpt = torch.load(wpath, map_location='cpu', weights_only=False)
        state = ckpt.get('state_dict', ckpt)
        load_res = model.load_state_dict(state, strict=False)
        print(f"    Missing keys: {len(load_res.missing_keys)}, Unexpected keys: {len(load_res.unexpected_keys)}")
        loaded_custom_ckpt = True
        break
if not loaded_custom_ckpt:
    print("[*] No custom checkpoint found (best.pt) — continuing with stock HuggingFace pretrained weights.")

if DEVICE == 'cuda':
    model = model.half().eval()
else:
    model = model.eval()

IMAGE_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=DEVICE, dtype=torch.float16 if DEVICE == 'cuda' else torch.float32).view(1, 3, 1, 1)
IMAGE_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=DEVICE, dtype=torch.float16 if DEVICE == 'cuda' else torch.float32).view(1, 3, 1, 1)

# Stream directly to .npy files on NVMe via memmap (0 GB extra system RAM overhead!)
out_9_path = '/content/encoded_keyframes_9tiles.npy'
out_1_path = '/content/encoded_globals.npy'
mem_9 = np.lib.format.open_memmap(out_9_path, mode='w+', dtype=np.float16, shape=(N, 9, 768))
mem_1 = np.lib.format.open_memmap(out_1_path, mode='w+', dtype=np.float16, shape=(N, 768))

t0 = time.time()
encoded_count = 0
log_interval = max(BATCH_SIZE * 4, 256)

for flat_uint8, batch_indices in loader:
    B = len(batch_indices)
    flat_tensor = flat_uint8.to(DEVICE, non_blocking=True)
    flat_tensor = flat_tensor.half() if DEVICE == 'cuda' else flat_tensor.float()
    flat_norm = (flat_tensor / 255.0).sub_(IMAGE_MEAN).div_(IMAGE_STD)

    with torch.inference_mode():
        feats = model.encode_image(flat_norm)
        feats = F.normalize(feats.float(), dim=-1).half()
        tiles_9 = feats.view(B, 9, 768).cpu().numpy()
        tiles_1 = tiles_9[:, 4, :]

    if batch_indices == list(range(encoded_count, encoded_count + B)):
        mem_9[encoded_count:encoded_count + B] = tiles_9
        mem_1[encoded_count:encoded_count + B] = tiles_1
    else:
        mem_9[batch_indices] = tiles_9
        mem_1[batch_indices] = tiles_1

    encoded_count += B

    if encoded_count % log_interval < B or encoded_count == N:
        mem_9.flush()
        mem_1.flush()
        elapsed = max(0.1, time.time() - t0)
        rate = encoded_count / elapsed
        eta_min = (N - encoded_count) / max(1e-3, rate) / 60.0
        print(f"[{encoded_count:,} / {N:,}] Speed: {rate:.1f} vids/s ({rate*9:.1f} frames/s) | ETA: {eta_min:.1f}m")

mem_9.flush()
mem_1.flush()
del mem_9, mem_1

print(f"\n[+] Finished encoding {N:,} videos in {(time.time() - t0)/60:.1f} mins!")

# Export matching manifest
out_manifest_path = '/content/encoded_manifest.json'
with open(out_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(items, f, indent=2)

# Copy outputs to Google Drive if mounted
drive_target = '/content/drive/MyDrive'
if os.path.exists(drive_target):
    print(f"[*] Copying ~2.05 GB of result files to Google Drive ({drive_target})...")
    shutil.copy(out_9_path, os.path.join(drive_target, 'encoded_keyframes_9tiles.npy'))
    shutil.copy(out_1_path, os.path.join(drive_target, 'encoded_globals.npy'))
    shutil.copy(out_manifest_path, os.path.join(drive_target, 'encoded_manifest.json'))
    print(f"[+] Copied all 3 result files directly to your Google Drive root ({drive_target})!")
else:
    print("[+] Files saved locally in /content/:")
    print("    - /content/encoded_keyframes_9tiles.npy")
    print("    - /content/encoded_globals.npy")
    print("    - /content/encoded_manifest.json")
    print("    (Download them directly from the Colab left file browser)")